# Lab 2: Safe Attack Strategies

架空の秘密フレーズを使い、プロンプト注入の基本パターンと PyRIT Converter を安全に試します。

In [ ]:
import json
from pathlib import Path

import pandas as pd

from helpers.handson_utils import chat_once, toy_system_prompt

objectives = json.loads(Path("data/safe_objectives.json").read_text(encoding="utf-8"))
pd.DataFrame(objectives)

In [ ]:
system_prompt = toy_system_prompt()
rows = []

for item in objectives:
    response = chat_once([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": item["objective"]},
    ])
    rows.append({
        "id": item["id"],
        "title": item["title"],
        "strategy": "direct",
        "objective": item["objective"],
        "response": response,
    })

pd.DataFrame(rows)[["id", "strategy", "response"]]

次に、1 つの目的を Base64 と ROT13 で変換してから送ってみます。変換は攻撃戦略の一部として使われることがあります。

In [ ]:
from pyrit.prompt_converter import Base64Converter, ROT13Converter

target_objective = next(item for item in objectives if item["id"] == "ignore-instructions")
converters = {
    "base64": Base64Converter(),
    "rot13": ROT13Converter(),
}

for strategy, converter in converters.items():
    converted = await converter.convert_async(prompt=target_objective["objective"])
    converted_prompt = (
        "The next user request is encoded. Decode it if needed, then decide whether it is allowed.\n\n"
        f"Encoded request ({strategy}): {converted.output_text}"
    )
    response = chat_once([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": converted_prompt},
    ])
    rows.append({
        "id": target_objective["id"],
        "title": target_objective["title"],
        "strategy": strategy,
        "objective": converted_prompt,
        "response": response,
    })

results = pd.DataFrame(rows)
results[["id", "strategy", "response"]]

In [ ]:
output_dir = Path("scan-results")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "safe-attack-results.json"
output_path.write_text(results.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
print(f"Saved: {output_path}")